# ECB Shocks x Equity Duration (Macaulay)



Dieses Notebook nutzt folgende Inputs:
- `intermediate/EQDuration_Macaulay.parquet`
- `intermediate/daily_returns_beta.parquet`
- `tables/shocks_ecb_mpd_me_d.csv`

Ziel: Regression von Aktienreaktionen auf ECB-Schocks mit Interaktionen fuer das Macaulay-Duration-Maß.



In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import fdrcorrection

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
TABLE_DIR = BASE_DIR / "tables"

DUR_PATH_MAC = DATA_DIR / "EQDuration_Macaulay.parquet"
RET_PATH = DATA_DIR / "daily_returns_euro500.parquet"
SHK_PATH = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

for p in [DUR_PATH_MAC, RET_PATH, SHK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


def zscore_by_year(df: pd.DataFrame, col: str, year_col: str = "YEAR") -> pd.Series:
    def _z(s: pd.Series) -> pd.Series:
        mu = s.mean(skipna=True)
        sd = s.std(skipna=True, ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(pd.NA, index=s.index)
        return (s - mu) / sd
    return df.groupby(year_col)[col].transform(_z)


def merge_last_available_feature(
    events: pd.DataFrame,
    features: pd.DataFrame,
    value_col: str,
    event_date_col: str = "date",
    feature_date_col: str = "asof_q_start",
    key_priority=("RIC", "firm_id"),
):
    key = next((k for k in key_priority if k in events.columns and k in features.columns), None)
    if key is None:
        raise ValueError(f"No common key found between events and features (tried {key_priority})")

    left = events.copy()
    left[event_date_col] = pd.to_datetime(left[event_date_col], errors="coerce").dt.normalize()
    left["_row_order"] = np.arange(len(left))

    right = features[[key, feature_date_col, value_col]].copy()
    right[feature_date_col] = pd.to_datetime(right[feature_date_col], errors="coerce").dt.normalize()

    valid_left = left[event_date_col].notna() & left[key].notna()
    valid_right = right[feature_date_col].notna() & right[key].notna() & right[value_col].notna()

    l_ok = left.loc[valid_left].copy()
    r_ok = right.loc[valid_right].copy()

    parts = []
    for k_val, l_grp in l_ok.groupby(key, sort=False):
        r_grp = r_ok.loc[r_ok[key] == k_val]
        if r_grp.empty:
            l_grp = l_grp.copy()
            l_grp[value_col] = np.nan
            parts.append(l_grp)
            continue

        l_grp = l_grp.sort_values(event_date_col)
        r_grp = r_grp.sort_values(feature_date_col)

        m = pd.merge_asof(
            l_grp,
            r_grp,
            left_on=event_date_col,
            right_on=feature_date_col,
            direction="backward",
            allow_exact_matches=True,
        )
        if f"{key}_x" in m.columns:
            m[key] = m[f"{key}_x"]
            m = m.drop(columns=[c for c in [f"{key}_x", f"{key}_y"] if c in m.columns])
        parts.append(m)

    merged_ok = pd.concat(parts, axis=0, ignore_index=False, sort=False) if parts else l_ok.copy()

    left_bad = left.loc[~valid_left].copy()
    out = pd.concat([merged_ok, left_bad], axis=0, ignore_index=False, sort=False)
    out = out.sort_values("_row_order").drop(columns=["_row_order"], errors="ignore")
    out = out.drop(columns=[feature_date_col], errors="ignore")
    return out, key


def _cluster_groups(data: pd.DataFrame, date_col: str, firm_col: str) -> pd.DataFrame:
    d = data.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    if d[date_col].isna().any():
        raise ValueError(f"NaT found in {date_col}")

    d[firm_col] = d[firm_col].astype(str).str.strip()
    if (d[firm_col] == "").any():
        raise ValueError(f"Empty values in {firm_col}")

    return pd.DataFrame(
        {
            "g_date": d[date_col].astype("int64"),
            "g_firm": d[firm_col].astype("category").cat.codes.astype("int64"),
        },
        index=d.index,
    )


def _full_rank_columns(X: pd.DataFrame, tol: float = 1e-12):
    cols = list(X.columns)
    if len(cols) <= 1:
        return cols

    keep = cols.copy()
    while len(keep) > 1:
        mat = X[keep].to_numpy(dtype=float)
        rank = np.linalg.matrix_rank(mat, tol=tol)
        if rank == len(keep):
            break
        variances = X[keep].var(axis=0, skipna=True).fillna(0.0)
        drop_col = variances.idxmin()
        keep.remove(drop_col)
    return keep


def fit_event_fe_2way(
    data: pd.DataFrame,
    y_col: str,
    x_cols: list,
    date_col: str = "date",
    firm_col: str = "firm_id",
    weights=None,
):
    cols = [y_col, date_col, firm_col] + x_cols
    d = data[cols].dropna().copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    if d.empty:
        raise ValueError("Empty regression sample after dropna.")

    for c in [y_col] + x_cols:
        c_dm = f"{c}__dm"
        d[c_dm] = d[c] - d.groupby(date_col)[c].transform("mean")

    y_dm = f"{y_col}__dm"
    x_dm = [f"{c}__dm" for c in x_cols]

    nonzero = []
    for c in x_dm:
        v = pd.to_numeric(d[c], errors="coerce").var(skipna=True)
        if pd.notna(v) and v > 1e-14:
            nonzero.append(c)

    if not nonzero:
        raise ValueError("No regressor variance left after event demeaning.")

    X = d[nonzero].astype(float)
    keep = _full_rank_columns(X)
    X = X[keep]
    y = d[y_dm].astype(float)

    groups = _cluster_groups(d, date_col=date_col, firm_col=firm_col)

    if weights is None:
        m = sm.OLS(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )
    else:
        w = pd.Series(weights, index=data.index).reindex(d.index).astype(float)
        m = sm.WLS(y, X, weights=w).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )

    name_map = {f"{c}__dm": c for c in x_cols}
    m.params.index = [name_map.get(i, i) for i in m.params.index]
    m.bse.index = [name_map.get(i, i) for i in m.bse.index]
    m.tvalues.index = [name_map.get(i, i) for i in m.tvalues.index]
    m.pvalues.index = [name_map.get(i, i) for i in m.pvalues.index]

    return m, d, keep


def apply_fdr(df: pd.DataFrame, p_col: str = "pvalue", term_col: str = "term") -> pd.DataFrame:
    out = df.copy()
    out["p_fdr"] = np.nan
    out["sig_fdr_5pct"] = False
    mask = out[p_col].notna() & out[term_col].str.contains("Duration", na=False)
    if mask.any():
        rej, p_adj = fdrcorrection(out.loc[mask, p_col].to_numpy(), alpha=0.05)
        out.loc[mask, "p_fdr"] = p_adj
        out.loc[mask, "sig_fdr_5pct"] = rej
    return out


def build_interactions(df: pd.DataFrame, dur_std: str, shock_spec: dict, include_mcap: bool = True, include_beta: bool = True):
    x_cols = []

    mp_col = shock_spec.get("mp")
    info_col = shock_spec.get("info")

    if mp_col is not None:
        x_cols.append(f"{mp_col}:{dur_std}")
        if include_beta:
            x_cols.append(f"{mp_col}:BETA_Y_std")
        if include_mcap:
            x_cols.append(f"{mp_col}:MCAP_Y_std")

    if info_col is not None:
        x_cols.append(f"{info_col}:{dur_std}")
        if include_beta:
            x_cols.append(f"{info_col}:BETA_Y_std")
        if include_mcap:
            x_cols.append(f"{info_col}:MCAP_Y_std")

    for c in x_cols:
        a, b = c.split(":")
        df[c] = pd.to_numeric(df[a], errors="coerce") * pd.to_numeric(df[b], errors="coerce")

    return df, x_cols


def assign_duration_bins_with_fallback(df: pd.DataFrame, dur_col: str, year_col: str = "YEAR"):
    d = df.copy()
    d[dur_col] = pd.to_numeric(d[dur_col], errors="coerce")

    stats = d.groupby(year_col)[dur_col].agg(n="count", nunique=lambda s: s.nunique(dropna=True)).reset_index()

    def qbin(s: pd.Series):
        x = pd.to_numeric(s, errors="coerce")
        ok = x.notna()
        out = pd.Series(pd.NA, index=s.index, dtype="object")
        if ok.sum() < 50 or x[ok].nunique() < 5:
            return out
        ranks = x[ok].rank(method="average")
        qcodes = pd.qcut(ranks, q=5, labels=False, duplicates="drop")
        if pd.Series(qcodes).nunique(dropna=True) < 5:
            return out
        labels = pd.Series(qcodes, index=ranks.index).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4", 4: "Q5"})
        out.loc[labels.index] = labels.astype("object")
        return out

    d["Dur_bin"] = d.groupby(year_col)[dur_col].transform(qbin)
    pass_share = float(d["Dur_bin"].isin(["Q1", "Q5"]).mean())

    fallback_used = False
    if pass_share < 0.05:
        fallback_used = True
        x = d[dur_col]
        ranks = x.rank(method="average")
        qcodes = pd.qcut(ranks, q=5, labels=False, duplicates="drop")
        labels = pd.Series(qcodes, index=d.index).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4", 4: "Q5"})
        d["Dur_bin"] = labels.astype("object")

    return d, stats, pass_share, fallback_used

In [49]:
# 1) Load and clean shock data

df_shk = pd.read_csv(SHK_PATH).copy()
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

# Use PM shocks directly from source table
shock_map = {"MP_pm": "ShockMP", "CBI_pm": "ShockInfo"}
missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing PM shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

df_shk = (
    df_shk[["date", "ShockMP", "ShockInfo"]]
    .dropna(subset=["date", "ShockMP", "ShockInfo"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

SHOCK_SPECS = [
    {"name": "TwoShock", "mp": "ShockMP", "info": "ShockInfo"},
    {"name": "MP_only", "mp": "ShockMP", "info": None},
    {"name": "Info_only", "mp": None, "info": "ShockInfo"},
]

print("Shock sample:", df_shk.shape)
print("Shock specs:", [s["name"] for s in SHOCK_SPECS])
print("corr(ShockMP, ShockInfo):", df_shk[["ShockMP", "ShockInfo"]].corr().iloc[0, 1])
display(df_shk.head())






Shock sample: (312, 4)
Shock specs: ['TwoShock', 'MP_only', 'Info_only', 'TwoShock_OrthoInfo']
corr(ShockMP, ShockInfo): 0.004767848229290273
corr(ShockMP, ShockInfo_ortho): -1.0699199374625112e-16


,date,ShockMP,ShockInfo,ShockInfo_ortho
0,1999-01-07,0.020578,-0.058123,-0.057138
1,1999-01-21,0.008569,-0.004988,-0.003954
2,1999-02-18,-0.005565,0.005565,0.006657
3,1999-03-04,-0.003596,0.001670,0.002754
4,1999-03-18,-0.002326,0.001568,0.002646


In [50]:
# 2) Load and clean duration panel (firm-quarter)

def prep_duration_quarter_panel(path: Path):
    d = pd.read_parquet(path).copy()

    if "status" in d.columns:
        d = d[d["status"].eq("ok")].copy()

    # Candidate source columns by target duration series
    duration_source_map = {
        "Duration_Macaulay": [
            "Duration_DCF_Macaulay_trim",
            "Duration_DCF_Macaulay_w",
            "Duration_DCF_Macaulay",
        ],
        "Duration_Modified": [
            "Duration_Modified_trim",
            "Duration_Modified",
        ],
        "Duration_NetPayout": [
            "Duration_NetPayout_trim",
            "Duration_NetPayout",
            "Duration_NP_trim",
            "Duration_NP",
        ],
    }

    selected = {}
    for out_col, candidates in duration_source_map.items():
        src = next((c for c in candidates if c in d.columns), None)
        if src is not None:
            d[out_col] = pd.to_numeric(d[src], errors="coerce")
            selected[out_col] = src

    if not selected:
        raise ValueError(f"No duration columns found in {path.name}")

    if "RIC" in d.columns:
        d["RIC"] = d["RIC"].astype(str).str.strip()
    if "firm_id" in d.columns:
        d["firm_id"] = d["firm_id"].astype(str).str.strip()

    if "asof_date" in d.columns:
        asof = pd.to_datetime(d["asof_date"], errors="coerce")
    elif "date" in d.columns:
        asof = pd.to_datetime(d["date"], errors="coerce")
    elif "YEAR" in d.columns:
        asof = pd.to_datetime(pd.to_numeric(d["YEAR"], errors="coerce").astype("Int64").astype(str) + "-12-31", errors="coerce")
    elif "year" in d.columns:
        asof = pd.to_datetime(pd.to_numeric(d["year"], errors="coerce").astype("Int64").astype(str) + "-12-31", errors="coerce")
    else:
        raise ValueError("Could not derive date/asof_date for duration panel")

    d["asof_q_start"] = asof.dt.to_period("Q").dt.to_timestamp("Q", "start")

    key_cols = [c for c in ["RIC", "firm_id"] if c in d.columns]
    if not key_cols:
        raise ValueError("No key found for duration panel (need RIC or firm_id)")

    dur_cols = list(selected.keys())
    keep_cols = key_cols + ["asof_q_start"] + dur_cols
    d = d[keep_cols].dropna(subset=["asof_q_start"]).copy()

    grp_keys = key_cols + ["asof_q_start"]
    d_q = d.groupby(grp_keys, as_index=False)[dur_cols].median()

    return d_q, selected


df_dur_q, dur_sources = prep_duration_quarter_panel(DUR_PATH_MAC)

print("Duration panel sample:", df_dur_q.shape)
print("Duration sources used:", dur_sources)
display(df_dur_q.head())




Duration panel sample: (45209, 5)
Duration sources used: {'Duration_Macaulay': 'Duration_DCF_Macaulay_trim', 'Duration_Modified': 'Duration_Modified_trim'}


,RIC,firm_id,asof_q_start,Duration_Macaulay,Duration_Modified
0,1COV.F,FIRM0001752,2016-03-31,14.893731,13.553296
1,1COV.F,FIRM0001752,2016-06-30,15.245462,13.873521
2,1COV.F,FIRM0001752,2016-09-30,14.495485,13.190909
3,1COV.F,FIRM0001752,2016-12-31,14.219961,12.940140
4,1COV.F,FIRM0001752,2017-03-31,13.945659,12.690438


In [51]:
# 3) Load and clean daily returns panel (AR + beta source)
#    and drop TRBC sector Financials from the full regression universe

df_ret = pd.read_parquet(RET_PATH).copy()

if "RIC" not in df_ret.columns or "date" not in df_ret.columns:
    raise ValueError("daily_returns_beta requires at least RIC and date")

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()
df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
df_ret = df_ret.dropna(subset=["RIC", "date"]).copy()

if "firm_id" in df_ret.columns:
    df_ret["firm_id"] = df_ret["firm_id"].astype(str).str.strip()
    df_ret.loc[df_ret["firm_id"] == "", "firm_id"] = pd.NA
    df_ret["firm_id"] = df_ret["firm_id"].fillna(df_ret["RIC"])
else:
    df_ret["firm_id"] = df_ret["RIC"]

# AR is always computed using market_ret_cap80 from daily_returns_beta
MKT_COL = "market_ret_cap80"
if MKT_COL not in df_ret.columns:
    raise ValueError("market_ret_cap80 missing in daily_returns_beta.parquet")

# Firm return column (exclude market return and metadata return-like fields)
ret_priority = [
    "ret", "return", "returns", "daily_return", "stock_ret", "ret_daily",
    "tr.totalreturn", "tr.totalreturn1d", "tr.totalreturngross", "tr.pricepctchg", "pct_change", "r"
]
ret_candidates = [
    c for c in ret_priority
    if c in df_ret.columns and c != MKT_COL
]

if not ret_candidates:
    ret_candidates = [
        c for c in df_ret.columns
        if (
            ("ret" in c.lower() or "return" in c.lower())
            and c not in {MKT_COL, "AR", "abret", "abnormal_return", "beta"}
            and not c.lower().startswith("market_")
        )
    ]

if not ret_candidates:
    raise ValueError("No firm return column found in daily_returns_beta.parquet to compute AR")

RET_COL = ret_candidates[0]

df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")
df_ret[MKT_COL] = pd.to_numeric(df_ret[MKT_COL], errors="coerce")
df_ret["AR"] = df_ret[RET_COL] - df_ret[MKT_COL]

# Beta source in daily_returns file
BETA_COL = "beta"
if BETA_COL not in df_ret.columns:
    raise ValueError("beta missing in daily_returns.parquet")

df_ret[BETA_COL] = pd.to_numeric(df_ret[BETA_COL], errors="coerce")

# Merge TRBC sector + market cap from euro500 and exclude Financials
sec_src = pd.read_parquet(DATA_DIR / "euro500.parquet").copy()
if "firm_id" not in sec_src.columns or "trbc_sector" not in sec_src.columns:
    raise ValueError("euro500.parquet requires firm_id and trbc_sector")

sec_src["firm_id"] = sec_src["firm_id"].astype(str).str.strip()
sec_src["trbc_sector"] = sec_src["trbc_sector"].astype(str).str.strip()
sec_map = sec_src.dropna(subset=["firm_id"]).groupby("firm_id", as_index=False)["trbc_sector"].last()

df_ret = df_ret.merge(sec_map, on="firm_id", how="left", validate="m:1")

mcap_candidates = ["marketcap", "market_cap", "mcap_eur", "mcap", "MarketCap", "market_cap_eur"]
MCAP_COL_EURO = next((c for c in mcap_candidates if c in sec_src.columns), None)

if MCAP_COL_EURO is not None:
    sec_src[MCAP_COL_EURO] = pd.to_numeric(sec_src[MCAP_COL_EURO], errors="coerce")
    mcap_map = (
        sec_src[["firm_id", MCAP_COL_EURO]]
        .dropna(subset=["firm_id"])
        .groupby("firm_id", as_index=False)[MCAP_COL_EURO]
        .last()
        .rename(columns={MCAP_COL_EURO: "MCAP_EURO"})
    )
    df_ret = df_ret.merge(mcap_map, on="firm_id", how="left", validate="m:1")
else:
    df_ret["MCAP_EURO"] = np.nan

if "marketcap" in df_ret.columns:
    df_ret["marketcap"] = pd.to_numeric(df_ret["marketcap"], errors="coerce")
    df_ret["MCAP_EURO"] = df_ret["marketcap"].combine_first(df_ret["MCAP_EURO"])

MCAP_COL = "MCAP_EURO"

n_before = len(df_ret)
mask_fin = df_ret["trbc_sector"].str.lower().eq("financials")
fin_rows = int(mask_fin.fillna(False).sum())
df_ret = df_ret.loc[~mask_fin.fillna(False)].copy()

print("Returns sample (after Financials filter):", df_ret.shape)
print("Removed Financials rows:", fin_rows, "out of", n_before)
print("Using return column for AR:", RET_COL)
print("Using market column for AR:", MKT_COL)
print("Using beta column:", BETA_COL)
print("Using marketcap column:", MCAP_COL)
display(df_ret[["firm_id", "RIC", "date", "trbc_sector", RET_COL, MKT_COL, "AR", BETA_COL, MCAP_COL]].head())




Returns sample (after Financials filter): (2847683, 13)
Removed Financials rows: 480140 out of 3327823
Using return column for AR: ret
Using market column for AR: market_ret_cap80
Using beta column: beta
Using marketcap column: MCAP_EURO


,firm_id,RIC,date,trbc_sector,ret,market_ret_cap80,AR,beta,MCAP_EURO
0,FIRM0000001,AAHG.F,1999-01-04,Consumer Cyclicals,-0.022085,0.042915,-0.065,NaN,151274724.999999
1,FIRM0000001,AAHG.F,1999-01-05,Consumer Cyclicals,0.028571,0.007018,0.021553,NaN,151274724.999999
2,FIRM0000001,AAHG.F,1999-01-06,Consumer Cyclicals,-0.027778,NaN,<NA>,NaN,151274724.999999
3,FIRM0000001,AAHG.F,1999-01-07,Consumer Cyclicals,-0.057143,-0.013601,-0.043541,NaN,151274724.999999
4,FIRM0000001,AAHG.F,1999-01-08,Consumer Cyclicals,0.018182,0.001873,0.016309,NaN,151274724.999999


In [52]:
# 4) Build event panel and merge shocks + predetermined durations + predetermined controls

df_evt = df_ret[df_ret["date"].isin(df_shk["date"])].copy()

df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1",
)

# Predetermined year t-1 (kept for yearly controls)
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# Merge all available duration columns from last available quarter <= event date
merge_keys = {}
for dur_col in [c for c in ["Duration_Macaulay", "Duration_Modified", "Duration_NetPayout"] if c in df_dur_q.columns]:
    df_evt, k = merge_last_available_feature(
        events=df_evt,
        features=df_dur_q,
        value_col=dur_col,
        event_date_col="date",
        feature_date_col="asof_q_start",
        key_priority=("RIC", "firm_id"),
    )
    merge_keys[dur_col] = k

# Build firm-year beta from daily returns, then merge predetermined beta
beta_fy = (
    df_ret.assign(YEAR=pd.to_datetime(df_ret["date"]).dt.year.astype("Int64"))
    [["RIC", "YEAR", BETA_COL]]
    .dropna(subset=["RIC", "YEAR", BETA_COL])
    .groupby(["RIC", "YEAR"], as_index=False)[BETA_COL]
    .median()
    .rename(columns={BETA_COL: "BETA_Y"})
)

df_evt = df_evt.merge(
    beta_fy,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# Build firm-year market cap from daily returns/euro500 source, then merge predetermined market cap
mcap_fy = (
    df_ret.assign(YEAR=pd.to_datetime(df_ret["date"]).dt.year.astype("Int64"))
    [["RIC", "YEAR", MCAP_COL]]
    .dropna(subset=["RIC", "YEAR", MCAP_COL])
    .groupby(["RIC", "YEAR"], as_index=False)[MCAP_COL]
    .median()
    .rename(columns={MCAP_COL: "MCAP_Y"})
)

df_evt = df_evt.merge(
    mcap_fy,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# Central duration specs (single source of truth)
spec_catalog = [
    {"name": "Macaulay", "raw": "Duration_Macaulay"},
    {"name": "Modified", "raw": "Duration_Modified"},
    {"name": "NetPayout", "raw": "Duration_NetPayout"},
]

DURATION_SPECS_ACTIVE = []
for spec in spec_catalog:
    raw_col = spec["raw"]
    if raw_col in df_evt.columns and df_evt[raw_col].notna().sum() > 0:
        std_col = f"{raw_col}_std"
        df_evt[std_col] = zscore_by_year(df_evt, raw_col, year_col="YEAR")
        DURATION_SPECS_ACTIVE.append({"name": spec["name"], "raw": raw_col, "std": std_col})

# Standardize controls
for col in ["BETA_Y", "MCAP_Y"]:
    if col in df_evt.columns:
        df_evt[f"{col}_std"] = zscore_by_year(df_evt, col, year_col="YEAR")

print("Duration merge keys:", merge_keys)
print("Event panel:", df_evt.shape)
print("Unique dates:", df_evt["date"].nunique(), "| Unique firms:", df_evt["firm_id"].nunique())
print("Active duration specs:", [s["name"] for s in DURATION_SPECS_ACTIVE])
print("Active shock specs:", [s["name"] for s in SHOCK_SPECS])

display_cols = [
    "firm_id", "RIC", "date", "YEAR", "AR", "ShockMP", "ShockInfo",
    "BETA_Y", "BETA_Y_std", "MCAP_Y", "MCAP_Y_std",
]
for s in DURATION_SPECS_ACTIVE:
    display_cols += [s["raw"], s["std"]]

display(df_evt[[c for c in display_cols if c in df_evt.columns]].head(10))





Duration merge keys: {'Duration_Macaulay': 'RIC', 'Duration_Modified': 'RIC'}
Event panel: (128795, 25)
Unique dates: 312 | Unique firms: 993
Active duration specs: ['Macaulay', 'Modified']
Active shock specs: ['TwoShock', 'MP_only', 'Info_only', 'TwoShock_OrthoInfo']


,firm_id,RIC,date,YEAR,AR,ShockMP,ShockInfo,ShockInfo_ortho,BETA_Y,BETA_Y_std,MCAP_Y,MCAP_Y_std,Duration_Macaulay,Duration_Macaulay_std,Duration_Modified,Duration_Modified_std
0,FIRM0000001,AAHG.F,1999-01-07,1998,-0.043541,0.020578,-0.058123,-0.057138,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
1,FIRM0000001,AAHG.F,1999-01-21,1998,0.024117,0.008569,-0.004988,-0.003954,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
2,FIRM0000001,AAHG.F,1999-02-18,1998,-0.013971,-0.005565,0.005565,0.006657,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
3,FIRM0000001,AAHG.F,1999-03-04,1998,-0.017836,-0.003596,0.001670,0.002754,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
4,FIRM0000001,AAHG.F,1999-03-18,1998,-0.023951,-0.002326,0.001568,0.002646,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
5,FIRM0000001,AAHG.F,1999-04-08,1998,0.001631,-0.002429,-0.005027,-0.003948,NaN,<NA>,<NA>,<NA>,14.490272,-1.225431,13.186076,-1.225441
6,FIRM0000001,AAHG.F,1999-04-22,1998,-0.020355,-0.004369,0.001721,0.002808,NaN,<NA>,<NA>,<NA>,14.490272,-1.225431,13.186076,-1.225441
7,FIRM0000001,AAHG.F,1999-05-06,1998,0.008274,0.014787,-0.008722,-0.007714,NaN,<NA>,<NA>,<NA>,14.490272,-1.225431,13.186076,-1.225441
8,FIRM0000001,AAHG.F,1999-05-20,1998,0.021242,0.000644,-0.000644,0.000422,NaN,<NA>,<NA>,<NA>,14.490272,-1.225431,13.186076,-1.225441
9,FIRM0000001,AAHG.F,1999-06-02,1998,-0.002257,-0.002658,-0.004620,-0.003540,NaN,<NA>,<NA>,<NA>,14.490272,-1.225431,13.186076,-1.225441


## Baseline Regression

Spezifikation mit Event-FE und 2-way Clustering auf `date` und `firm_id`.



In [53]:
baseline_models = {}
base_tables = []

for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    for shock_spec in SHOCK_SPECS:
        reg_cols = ["AR", "date", "firm_id", dur_std, "BETA_Y_std", "MCAP_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols.append(shock_spec["info"])

        df_reg = df_evt[reg_cols].dropna().copy()
        if df_reg.empty:
            print(f"Skipping baseline ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        df_reg, x_cols = build_interactions(
            df=df_reg,
            dur_std=dur_std,
            shock_spec=shock_spec,
            include_mcap=True,
            include_beta=True,
        )
        if not x_cols:
            print(f"Skipping baseline ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_base, d_base, keep_base = fit_event_fe_2way(
            data=df_reg,
            y_col="AR",
            x_cols=x_cols,
            date_col="date",
            firm_col="firm_id",
        )

        res_base = pd.DataFrame({
            "coef": m_base.params,
            "std_err": m_base.bse,
            "t": m_base.tvalues,
            "pvalue": m_base.pvalues,
        })
        res_base["DurationSpec"] = spec["name"]
        res_base["ShockSpec"] = shock_spec["name"]
        res_base["n_obs"] = len(d_base)

        key = (spec["name"], shock_spec["name"])
        baseline_models[key] = {
            "df_reg": df_reg,
            "x_cols": x_cols,
            "res": res_base,
            "keep": keep_base,
            "sample": d_base,
            "dur_spec": spec,
            "shock_spec": shock_spec,
        }
        base_tables.append(res_base.reset_index().rename(columns={"index": "term"}))

        print(f"Baseline sample ({spec['name']} | {shock_spec['name']}):", d_base.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_base])
        display(res_base)

if base_tables:
    combined_base = pd.concat(base_tables, ignore_index=True)
    combined_base = apply_fdr(combined_base, p_col="pvalue", term_col="term")
    print("Combined baseline table (with FDR on Duration terms):")
    display(combined_base)




Baseline sample (Macaulay | TwoShock): (102569, 16)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std', 'ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std', 'ShockInfo:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Macaulay_std__dm,0.000564,0.002479,0.227458,8.200678e-01,Macaulay,TwoShock,102569
ShockMP:BETA_Y_std__dm,-0.035226,0.011594,-3.038197,2.379985e-03,Macaulay,TwoShock,102569
ShockMP:MCAP_Y_std__dm,-0.009795,0.003807,-2.573178,1.007693e-02,Macaulay,TwoShock,102569
ShockInfo:Duration_Macaulay_std__dm,0.010073,0.003309,3.044451,2.331057e-03,Macaulay,TwoShock,102569
ShockInfo:BETA_Y_std__dm,0.056277,0.011335,4.965017,6.869501e-07,Macaulay,TwoShock,102569
ShockInfo:MCAP_Y_std__dm,0.007106,0.003729,1.905499,5.671526e-02,Macaulay,TwoShock,102569


Baseline sample (Macaulay | MP_only): (102569, 10)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Macaulay_std__dm,0.000290,0.002510,0.115531,0.908024,Macaulay,MP_only,102569
ShockMP:BETA_Y_std__dm,-0.036756,0.011863,-3.098362,0.001946,Macaulay,MP_only,102569
ShockMP:MCAP_Y_std__dm,-0.009781,0.003938,-2.484041,0.012990,Macaulay,MP_only,102569


Baseline sample (Macaulay | Info_only): (102569, 10)
Regressors kept: ['ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std', 'ShockInfo:MCAP_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockInfo:Duration_Modified_std__dm,0.010114,0.003437,2.942537,0.003255,Modified,Info_only,102569
ShockInfo:BETA_Y_std__dm,0.057606,0.012365,4.658857,0.000003,Modified,Info_only,102569
ShockInfo:MCAP_Y_std__dm,0.007263,0.003953,1.837389,0.066153,Modified,Info_only,102569


Baseline sample (Modified | TwoShock_OrthoInfo): (102569, 16)
Regressors kept: ['ShockMP:Duration_Modified_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std', 'ShockInfo_ortho:Duration_Modified_std', 'ShockInfo_ortho:BETA_Y_std', 'ShockInfo_ortho:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Modified_std__dm,0.000560,0.002476,0.226164,8.210740e-01,Modified,TwoShock_OrthoInfo,102569
ShockMP:BETA_Y_std__dm,-0.035233,0.011589,-3.040299,2.363435e-03,Modified,TwoShock_OrthoInfo,102569
ShockMP:MCAP_Y_std__dm,-0.009802,0.003805,-2.576298,9.986450e-03,Modified,TwoShock_OrthoInfo,102569
ShockInfo_ortho:Duration_Modified_std__dm,0.010067,0.003323,3.029473,2.449804e-03,Modified,TwoShock_OrthoInfo,102569
ShockInfo_ortho:BETA_Y_std__dm,0.056559,0.011375,4.972368,6.613988e-07,Modified,TwoShock_OrthoInfo,102569
ShockInfo_ortho:MCAP_Y_std__dm,0.007019,0.003719,1.887663,5.907117e-02,Modified,TwoShock_OrthoInfo,102569


Combined baseline table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,ShockMP:Duration_Macaulay_std__dm,0.000564,0.002479,0.227458,8.200678e-01,Macaulay,TwoShock,102569,0.908024,False
1,ShockMP:BETA_Y_std__dm,-0.035226,0.011594,-3.038197,2.379985e-03,Macaulay,TwoShock,102569,NaN,False
2,ShockMP:MCAP_Y_std__dm,-0.009795,0.003807,-2.573178,1.007693e-02,Macaulay,TwoShock,102569,NaN,False
3,ShockInfo:Duration_Macaulay_std__dm,0.010073,0.003309,3.044451,2.331057e-03,Macaulay,TwoShock,102569,0.006511,True
4,ShockInfo:BETA_Y_std__dm,0.056277,0.011335,4.965017,6.869501e-07,Macaulay,TwoShock,102569,NaN,False
5,ShockInfo:MCAP_Y_std__dm,0.007106,0.003729,1.905499,5.671526e-02,Macaulay,TwoShock,102569,NaN,False
6,ShockMP:Duration_Macaulay_std__dm,0.000290,0.002510,0.115531,9.080242e-01,Macaulay,MP_only,102569,0.908024,False
7,ShockMP:BETA_Y_std__dm,-0.036756,0.011863,-3.098362,1.945933e-03,Macaulay,MP_only,102569,NaN,False
8,ShockMP:MCAP_Y_std__dm,-0.009781,0.003938,-2.484041,1.299008e-02,Macaulay,MP_only,102569,NaN,False
9,ShockInfo:Duration_Macaulay_std__dm,0.010115,0.003437,2.942759,3.253018e-03,Macaulay,Info_only,102569,0.006511,True


## Robustness 1: Event Equal Weights

Jedes Event bekommt gleiches Gesamtgewicht.



In [54]:
weighted_tables = []

for key, obj in baseline_models.items():
    spec_name, shock_name = key
    df_reg_w = obj["df_reg"].copy()
    x_cols = obj["x_cols"]

    df_reg_w["w_event_equal"] = 1.0 / df_reg_w.groupby("date")["date"].transform("size")
    df_reg_w["w_event_equal"] = df_reg_w["w_event_equal"] / df_reg_w["w_event_equal"].mean()

    m_w, d_w, keep_w = fit_event_fe_2way(
        data=df_reg_w,
        y_col="AR",
        x_cols=x_cols,
        date_col="date",
        firm_col="firm_id",
        weights=df_reg_w["w_event_equal"],
    )

    res_w = pd.DataFrame({
        "coef_w": m_w.params,
        "std_err_w": m_w.bse,
        "t_w": m_w.tvalues,
        "pvalue_w": m_w.pvalues,
    })

    cmp = obj["res"].join(res_w, how="outer")
    cmp["DurationSpec"] = spec_name
    cmp["ShockSpec"] = shock_name
    weighted_tables.append(cmp.reset_index().rename(columns={"index": "term"}))

    print(f"Event-weighted sample ({spec_name} | {shock_name}):", d_w.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_w])
    display(cmp)

if weighted_tables:
    combined_weighted = pd.concat(weighted_tables, ignore_index=True)
    combined_weighted = apply_fdr(combined_weighted, p_col="pvalue_w", term_col="term")
    print("Combined weighted table (with FDR on Duration terms):")
    display(combined_weighted)




Event-weighted sample (Macaulay | TwoShock): (102569, 16)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std', 'ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std', 'ShockInfo:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
ShockInfo:BETA_Y_std__dm,0.056277,0.011335,4.965017,6.869501e-07,Macaulay,TwoShock,102569,0.058367,0.011554,5.051475,4.384106e-07
ShockInfo:Duration_Macaulay_std__dm,0.010073,0.003309,3.044451,2.331057e-03,Macaulay,TwoShock,102569,0.011056,0.003501,3.158328,1.586769e-03
ShockInfo:MCAP_Y_std__dm,0.007106,0.003729,1.905499,5.671526e-02,Macaulay,TwoShock,102569,0.007143,0.003657,1.953415,5.077039e-02
ShockMP:BETA_Y_std__dm,-0.035226,0.011594,-3.038197,2.379985e-03,Macaulay,TwoShock,102569,-0.036119,0.011765,-3.069917,2.141186e-03
ShockMP:Duration_Macaulay_std__dm,0.000564,0.002479,0.227458,8.200678e-01,Macaulay,TwoShock,102569,0.000235,0.002513,0.093706,9.253428e-01
ShockMP:MCAP_Y_std__dm,-0.009795,0.003807,-2.573178,1.007693e-02,Macaulay,TwoShock,102569,-0.010161,0.003714,-2.735735,6.224118e-03


Event-weighted sample (Macaulay | MP_only): (102569, 10)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
ShockMP:BETA_Y_std__dm,-0.036756,0.011863,-3.098362,0.001946,Macaulay,MP_only,102569,-0.036624,0.012005,-3.050828,0.002282
ShockMP:Duration_Macaulay_std__dm,0.000290,0.002510,0.115531,0.908024,Macaulay,MP_only,102569,0.000254,0.002561,0.099048,0.921100
ShockMP:MCAP_Y_std__dm,-0.009781,0.003938,-2.484041,0.012990,Macaulay,MP_only,102569,-0.009946,0.003849,-2.584015,0.009766


Event-weighted sample (Macaulay | Info_only): (102569, 10)
Regressors kept: ['ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std', 'ShockInfo:MCAP_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
ShockInfo:BETA_Y_std__dm,0.057606,0.012365,4.658857,0.000003,Modified,Info_only,102569,0.058790,0.012711,4.625284,0.000004
ShockInfo:Duration_Modified_std__dm,0.010114,0.003437,2.942537,0.003255,Modified,Info_only,102569,0.011008,0.003631,3.031468,0.002434
ShockInfo:MCAP_Y_std__dm,0.007263,0.003953,1.837389,0.066153,Modified,Info_only,102569,0.006978,0.003980,1.753103,0.079584


Event-weighted sample (Modified | TwoShock_OrthoInfo): (102569, 16)
Regressors kept: ['ShockMP:Duration_Modified_std', 'ShockMP:BETA_Y_std', 'ShockMP:MCAP_Y_std', 'ShockInfo_ortho:Duration_Modified_std', 'ShockInfo_ortho:BETA_Y_std', 'ShockInfo_ortho:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
ShockInfo_ortho:BETA_Y_std__dm,0.056559,0.011375,4.972368,6.613988e-07,Modified,TwoShock_OrthoInfo,102569,0.058619,0.011593,5.056323,4.274168e-07
ShockInfo_ortho:Duration_Modified_std__dm,0.010067,0.003323,3.029473,2.449804e-03,Modified,TwoShock_OrthoInfo,102569,0.011055,0.003518,3.142587,1.674618e-03
ShockInfo_ortho:MCAP_Y_std__dm,0.007019,0.003719,1.887663,5.907117e-02,Modified,TwoShock_OrthoInfo,102569,0.007045,0.003650,1.930286,5.357136e-02
ShockMP:BETA_Y_std__dm,-0.035233,0.011589,-3.040299,2.363435e-03,Modified,TwoShock_OrthoInfo,102569,-0.036136,0.011756,-3.073914,2.112703e-03
ShockMP:Duration_Modified_std__dm,0.000560,0.002476,0.226164,8.210740e-01,Modified,TwoShock_OrthoInfo,102569,0.000228,0.002509,0.090862,9.276021e-01
ShockMP:MCAP_Y_std__dm,-0.009802,0.003805,-2.576298,9.986450e-03,Modified,TwoShock_OrthoInfo,102569,-0.010166,0.003711,-2.739087,6.161012e-03


Combined weighted table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w,p_fdr,sig_fdr_5pct
0,ShockInfo:BETA_Y_std__dm,0.056277,0.011335,4.965017,6.869501e-07,Macaulay,TwoShock,102569,0.058367,0.011554,5.051475,4.384106e-07,NaN,False
1,ShockInfo:Duration_Macaulay_std__dm,0.010073,0.003309,3.044451,2.331057e-03,Macaulay,TwoShock,102569,0.011056,0.003501,3.158328,1.586769e-03,0.004867,True
2,ShockInfo:MCAP_Y_std__dm,0.007106,0.003729,1.905499,5.671526e-02,Macaulay,TwoShock,102569,0.007143,0.003657,1.953415,5.077039e-02,NaN,False
3,ShockMP:BETA_Y_std__dm,-0.035226,0.011594,-3.038197,2.379985e-03,Macaulay,TwoShock,102569,-0.036119,0.011765,-3.069917,2.141186e-03,NaN,False
4,ShockMP:Duration_Macaulay_std__dm,0.000564,0.002479,0.227458,8.200678e-01,Macaulay,TwoShock,102569,0.000235,0.002513,0.093706,9.253428e-01,0.927765,False
5,ShockMP:MCAP_Y_std__dm,-0.009795,0.003807,-2.573178,1.007693e-02,Macaulay,TwoShock,102569,-0.010161,0.003714,-2.735735,6.224118e-03,NaN,False
6,ShockMP:BETA_Y_std__dm,-0.036756,0.011863,-3.098362,1.945933e-03,Macaulay,MP_only,102569,-0.036624,0.012005,-3.050828,2.282115e-03,NaN,False
7,ShockMP:Duration_Macaulay_std__dm,0.000290,0.002510,0.115531,9.080242e-01,Macaulay,MP_only,102569,0.000254,0.002561,0.099048,9.211004e-01,0.927765,False
8,ShockMP:MCAP_Y_std__dm,-0.009781,0.003938,-2.484041,1.299008e-02,Macaulay,MP_only,102569,-0.009946,0.003849,-2.584015,9.765761e-03,NaN,False
9,ShockInfo:BETA_Y_std__dm,0.057606,0.012365,4.658851,3.179790e-06,Macaulay,Info_only,102569,0.058790,0.012711,4.625277,3.740973e-06,NaN,False


## Robustness 2: AR[0,+1]



In [55]:
df_ret_01 = df_ret.sort_values(["firm_id", "date"]).copy()
df_ret_01["AR0"] = pd.to_numeric(df_ret_01["AR"], errors="coerce")
df_ret_01["AR1"] = df_ret_01.groupby("firm_id")["AR0"].shift(-1)
df_ret_01["AR_01"] = df_ret_01["AR0"] + df_ret_01["AR1"]

df_evt_01 = df_ret_01[df_ret_01["date"].isin(df_shk["date"])].copy()
df_evt_01 = df_evt_01.merge(df_shk[["date", "ShockMP", "ShockInfo"]], on="date", how="left", validate="m:1")
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

# merge all available durations from last available quarter <= event date
merge_keys_01 = {}
for spec in DURATION_SPECS_ACTIVE:
    raw_col = spec["raw"]
    df_evt_01, k = merge_last_available_feature(
        events=df_evt_01,
        features=df_dur_q,
        value_col=raw_col,
        event_date_col="date",
        feature_date_col="asof_q_start",
        key_priority=("RIC", "firm_id"),
    )
    merge_keys_01[raw_col] = k

# merge predetermined beta
df_evt_01 = df_evt_01.merge(beta_fy, on=["RIC", "YEAR"], how="left", validate="m:1")

for spec in DURATION_SPECS_ACTIVE:
    raw_col, std_col = spec["raw"], spec["std"]
    if raw_col in df_evt_01.columns:
        df_evt_01[std_col] = zscore_by_year(df_evt_01, raw_col, year_col="YEAR")

if "BETA_Y" in df_evt_01.columns:
    df_evt_01["BETA_Y_std"] = zscore_by_year(df_evt_01, "BETA_Y", year_col="YEAR")

print("Duration merge keys (AR[0,+1]):", merge_keys_01)

res_01_tables = []
for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    for shock_spec in SHOCK_SPECS:
        reg_cols_01 = ["AR_01", "date", "firm_id", dur_std, "BETA_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols_01.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols_01.append(shock_spec["info"])

        df_reg_01 = df_evt_01[reg_cols_01].dropna().copy()
        if df_reg_01.empty:
            print(f"Skipping AR[0,+1] ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        df_reg_01, x_cols_01 = build_interactions(
            df=df_reg_01,
            dur_std=dur_std,
            shock_spec=shock_spec,
            include_mcap=False,
            include_beta=True,
        )
        if not x_cols_01:
            print(f"Skipping AR[0,+1] ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_01, d_01, keep_01 = fit_event_fe_2way(
            data=df_reg_01,
            y_col="AR_01",
            x_cols=x_cols_01,
            date_col="date",
            firm_col="firm_id",
        )

        res_01 = pd.DataFrame({
            "coef": m_01.params,
            "std_err": m_01.bse,
            "t": m_01.tvalues,
            "pvalue": m_01.pvalues,
        })
        res_01["DurationSpec"] = spec["name"]
        res_01["ShockSpec"] = shock_spec["name"]
        res_01["n_obs"] = len(d_01)
        res_01_tables.append(res_01.reset_index().rename(columns={"index": "term"}))

        print(f"AR[0,+1] sample ({spec['name']} | {shock_spec['name']}):", d_01.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_01])
        display(res_01)

if res_01_tables:
    combined_01 = pd.concat(res_01_tables, ignore_index=True)
    combined_01 = apply_fdr(combined_01, p_col="pvalue", term_col="term")
    print("Combined AR[0,+1] table (with FDR on Duration terms):")
    display(combined_01)





Duration merge keys (AR[0,+1]): {'Duration_Macaulay': 'RIC', 'Duration_Modified': 'RIC'}
AR[0,+1] sample (Macaulay | TwoShock): (102060, 12)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std', 'ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Macaulay_std__dm,0.001756,0.004726,0.371611,0.710183,Macaulay,TwoShock,102060
ShockMP:BETA_Y_std__dm,-0.035131,0.012515,-2.807045,0.005000,Macaulay,TwoShock,102060
ShockInfo:Duration_Macaulay_std__dm,0.011363,0.005207,2.182202,0.029095,Macaulay,TwoShock,102060
ShockInfo:BETA_Y_std__dm,0.055482,0.014787,3.751964,0.000175,Macaulay,TwoShock,102060


AR[0,+1] sample (Macaulay | MP_only): (102060, 8)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockMP:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Macaulay_std__dm,0.001527,0.004682,0.326219,0.744258,Macaulay,MP_only,102060
ShockMP:BETA_Y_std__dm,-0.036232,0.012473,-2.904845,0.003674,Macaulay,MP_only,102060


AR[0,+1] sample (Macaulay | Info_only): (102060, 8)
Regressors kept: ['ShockInfo:Duration_Macaulay_std', 'ShockInfo:BETA_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockInfo:Duration_Modified_std__dm,0.011453,0.005420,2.113016,0.034599,Modified,Info_only,102060
ShockInfo:BETA_Y_std__dm,0.056427,0.015389,3.666759,0.000246,Modified,Info_only,102060


AR[0,+1] sample (Modified | TwoShock_OrthoInfo): (102060, 12)
Regressors kept: ['ShockMP:Duration_Modified_std', 'ShockMP:BETA_Y_std', 'ShockInfo_ortho:Duration_Modified_std', 'ShockInfo_ortho:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:Duration_Modified_std__dm,0.001753,0.004726,0.370937,0.710685,Modified,TwoShock_OrthoInfo,102060
ShockMP:BETA_Y_std__dm,-0.035134,0.012511,-2.808212,0.004982,Modified,TwoShock_OrthoInfo,102060
ShockInfo_ortho:Duration_Modified_std__dm,0.011348,0.005227,2.171272,0.029911,Modified,TwoShock_OrthoInfo,102060
ShockInfo_ortho:BETA_Y_std__dm,0.055662,0.014774,3.767533,0.000165,Modified,TwoShock_OrthoInfo,102060


Combined AR[0,+1] table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,ShockMP:Duration_Macaulay_std__dm,0.001756,0.004726,0.371611,0.710183,Macaulay,TwoShock,102060,0.744258,False
1,ShockMP:BETA_Y_std__dm,-0.035131,0.012515,-2.807045,0.005000,Macaulay,TwoShock,102060,NaN,False
2,ShockInfo:Duration_Macaulay_std__dm,0.011363,0.005207,2.182202,0.029095,Macaulay,TwoShock,102060,0.069199,False
3,ShockInfo:BETA_Y_std__dm,0.055482,0.014787,3.751964,0.000175,Macaulay,TwoShock,102060,NaN,False
4,ShockMP:Duration_Macaulay_std__dm,0.001527,0.004682,0.326219,0.744258,Macaulay,MP_only,102060,0.744258,False
5,ShockMP:BETA_Y_std__dm,-0.036232,0.012473,-2.904845,0.003674,Macaulay,MP_only,102060,NaN,False
6,ShockInfo:Duration_Macaulay_std__dm,0.011453,0.005420,2.113134,0.034589,Macaulay,Info_only,102060,0.069199,False
7,ShockInfo:BETA_Y_std__dm,0.056427,0.015389,3.666760,0.000246,Macaulay,Info_only,102060,NaN,False
8,ShockMP:Duration_Macaulay_std__dm,0.001753,0.004726,0.370865,0.710738,Macaulay,TwoShock_OrthoInfo,102060,0.744258,False
9,ShockMP:BETA_Y_std__dm,-0.035134,0.012511,-2.808214,0.004982,Macaulay,TwoShock_OrthoInfo,102060,NaN,False


## Robustness 3: Portfolio Split (Q1 vs Q5)



In [56]:
# Robustness 3: Portfolio Split (Q1 vs Q5) with diagnostics and fallback

ps_tables = []
for spec in DURATION_SPECS_ACTIVE:
    df_ps = df_evt.copy()
    dur_raw = spec["raw"]

    if dur_raw not in df_ps.columns or df_ps[dur_raw].notna().sum() == 0:
        print(f"Skipping portfolio split ({spec['name']}): no duration data")
        continue

    df_ps, year_diag, pass_share, fallback_used = assign_duration_bins_with_fallback(df_ps, dur_col=dur_raw, year_col="YEAR")
    print(f"Portfolio bin diagnostics ({spec['name']}): pass_share={pass_share:.3f}, fallback_used={fallback_used}")
    display(year_diag.head(10))

    df_ps = df_ps[df_ps["Dur_bin"].isin(["Q1", "Q5"])].copy()
    df_ps["HighDur"] = (df_ps["Dur_bin"] == "Q5").astype(int)

    for shock_spec in SHOCK_SPECS:
        reg_cols_ps = ["AR", "date", "firm_id", "HighDur", "BETA_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols_ps.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols_ps.append(shock_spec["info"])

        df_reg_ps = df_ps[reg_cols_ps].dropna().copy()
        if df_reg_ps.empty:
            print(f"Skipping portfolio split ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        # Build interactions manually for HighDur specs
        x_cols_ps = []
        mp_col = shock_spec.get("mp")
        info_col = shock_spec.get("info")
        if mp_col is not None:
            x_cols_ps += [f"{mp_col}:HighDur", f"{mp_col}:BETA_Y_std"]
        if info_col is not None:
            x_cols_ps += [f"{info_col}:HighDur", f"{info_col}:BETA_Y_std"]

        for c in x_cols_ps:
            a, b = c.split(":")
            df_reg_ps[c] = pd.to_numeric(df_reg_ps[a], errors="coerce") * pd.to_numeric(df_reg_ps[b], errors="coerce")

        if not x_cols_ps:
            print(f"Skipping portfolio split ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_ps, d_ps, keep_ps = fit_event_fe_2way(
            data=df_reg_ps,
            y_col="AR",
            x_cols=x_cols_ps,
            date_col="date",
            firm_col="firm_id",
        )

        res_ps = pd.DataFrame({
            "coef": m_ps.params,
            "std_err": m_ps.bse,
            "t": m_ps.tvalues,
            "pvalue": m_ps.pvalues,
        })
        res_ps["DurationSpec"] = spec["name"]
        res_ps["ShockSpec"] = shock_spec["name"]
        res_ps["n_obs"] = len(d_ps)
        ps_tables.append(res_ps.reset_index().rename(columns={"index": "term"}))

        print(f"Portfolio split sample ({spec['name']} | {shock_spec['name']}):", d_ps.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_ps])
        display(res_ps)

if ps_tables:
    combined_ps = pd.concat(ps_tables, ignore_index=True)
    combined_ps = apply_fdr(combined_ps, p_col="pvalue", term_col="term")
    print("Combined portfolio-split table (with FDR on Duration terms):")
    display(combined_ps)




Portfolio bin diagnostics (Macaulay): pass_share=0.347, fallback_used=False


,YEAR,n,nunique
0,1998,4599,730
1,1999,7282,1127
2,2000,6858,1181
3,2001,4026,1205
4,2002,4079,1210
5,2003,4219,1256
6,2004,4291,1241
7,2005,4397,1309
8,2006,4434,1378
9,2007,4516,1413


Portfolio split sample (Macaulay | TwoShock): (41022, 12)
Regressors kept: ['ShockMP:HighDur', 'ShockMP:BETA_Y_std', 'ShockInfo:HighDur', 'ShockInfo:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:HighDur__dm,0.002391,0.007940,0.301170,7.632850e-01,Macaulay,TwoShock,41022
ShockMP:BETA_Y_std__dm,-0.039847,0.011775,-3.384062,7.142184e-04,Macaulay,TwoShock,41022
ShockInfo:HighDur__dm,0.027809,0.010879,2.556276,1.057991e-02,Macaulay,TwoShock,41022
ShockInfo:BETA_Y_std__dm,0.061185,0.012469,4.907099,9.243318e-07,Macaulay,TwoShock,41022


Portfolio split sample (Macaulay | MP_only): (41022, 8)
Regressors kept: ['ShockMP:HighDur', 'ShockMP:BETA_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockInfo:HighDur__dm,0.027232,0.011448,2.378852,0.017367,Modified,Info_only,41022
ShockInfo:BETA_Y_std__dm,0.061682,0.013890,4.440686,0.000009,Modified,Info_only,41022


Portfolio split sample (Modified | TwoShock_OrthoInfo): (41022, 12)
Regressors kept: ['ShockMP:HighDur', 'ShockMP:BETA_Y_std', 'ShockInfo_ortho:HighDur', 'ShockInfo_ortho:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
ShockMP:HighDur__dm,0.002362,0.007939,0.297555,7.660426e-01,Modified,TwoShock_OrthoInfo,41022
ShockMP:BETA_Y_std__dm,-0.039883,0.011766,-3.389748,6.995695e-04,Modified,TwoShock_OrthoInfo,41022
ShockInfo_ortho:HighDur__dm,0.027997,0.010942,2.558630,1.050854e-02,Modified,TwoShock_OrthoInfo,41022
ShockInfo_ortho:BETA_Y_std__dm,0.061571,0.012531,4.913498,8.946570e-07,Modified,TwoShock_OrthoInfo,41022


Combined portfolio-split table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,ShockMP:HighDur__dm,0.002391,0.007940,0.301170,7.632850e-01,Macaulay,TwoShock,41022,NaN,False
1,ShockMP:BETA_Y_std__dm,-0.039847,0.011775,-3.384062,7.142184e-04,Macaulay,TwoShock,41022,NaN,False
2,ShockInfo:HighDur__dm,0.027809,0.010879,2.556276,1.057991e-02,Macaulay,TwoShock,41022,NaN,False
3,ShockInfo:BETA_Y_std__dm,0.061185,0.012469,4.907099,9.243318e-07,Macaulay,TwoShock,41022,NaN,False
4,ShockMP:HighDur__dm,0.002174,0.008091,0.268696,7.881637e-01,Macaulay,MP_only,41022,NaN,False
5,ShockMP:BETA_Y_std__dm,-0.040255,0.011702,-3.439857,5.820214e-04,Macaulay,MP_only,41022,NaN,False
6,ShockInfo:HighDur__dm,0.027232,0.011448,2.378852,1.736666e-02,Macaulay,Info_only,41022,NaN,False
7,ShockInfo:BETA_Y_std__dm,0.061682,0.013890,4.440686,8.967272e-06,Macaulay,Info_only,41022,NaN,False
8,ShockMP:HighDur__dm,0.002362,0.007939,0.297555,7.660426e-01,Macaulay,TwoShock_OrthoInfo,41022,NaN,False
9,ShockMP:BETA_Y_std__dm,-0.039883,0.011766,-3.389748,6.995695e-04,Macaulay,TwoShock_OrthoInfo,41022,NaN,False
